# Auto-Encoding Variational Bayes — figure reproduction (baseline models)

Reproduces **Figures 2–5** of Kingma & Welling, *Auto-Encoding Variational Bayes*
([arXiv:1312.6114](https://arxiv.org/abs/1312.6114)) using the **exact upstream VAE code**,
kept byte-for-byte in this folder:

Project layout (this notebook lives in `notebooks/`; the models are a sibling under `models/`):

```
survey/src/related_work/vae/
├── models/baseline/{novicestone_vae.py, pytorch_vae.py}   # exact upstream code
├── models/custom/                                         # (your own models later)
└── notebooks/aevb_vae_figures.ipynb                       # this notebook
```

- **`models/baseline/novicestone_vae.py`** — verbatim [NoviceStone/VAE](https://github.com/NoviceStone/VAE/blob/master/models.py)
  `models.py`. `VAE(input_size, hidden_size, latent_size, data_type)` — parameterizable,
  tanh, Bernoulli (`"binary"`) **or** Gaussian (`"real"`) decoder.
- **`models/baseline/pytorch_vae.py`** — verbatim [pytorch/examples](https://github.com/pytorch/examples/blob/main/vae/main.py)
  `vae/main.py`. Its `VAE` is **hardcoded to 784→400→20, MNIST/Bernoulli**.

We load the exact `VAE` classes (and the pytorch `loss_function`) by **AST extraction** — the
class text runs verbatim, but `main.py`'s module-level `argparse`/MNIST-download/training
scaffolding is *not* executed (it can't be imported cleanly into a notebook).

### Consequence of using the exact code
The upstream pytorch model cannot change its latent size, so:

| Figure | Model used | Why |
|---|---|---|
| 2 (Nz sweep) | NoviceStone sweep **+** one pytorch curve at Nz=20 | pytorch latent is fixed at 20 |
| 3 (Nz=3) | NoviceStone | pytorch can't do Nz=3 |
| 4 (Nz=2 manifold) | NoviceStone (MNIST + Frey) | pytorch can't do Nz=2 |
| 5 (Nz∈{2,5,10,20}) | NoviceStone | pytorch can only produce the Nz=20 panel |

### Honest deviations
1. **No wake-sleep / MC-EM baselines** (not in either repo) → Figure 2 overlays the two
   implementations instead.
2. **Figure 3** uses an importance-weighted marginal-likelihood estimate, not the paper's HMC.
3. NoviceStone's `models.py` ships **no loss function**, so we compute the standard ELBO that
   matches its decoder; the pytorch path trains with its own upstream `loss_function`.


## 0. Where it runs & dependencies

Runs **locally**, on **Kaggle**, and on **Colab**. Kaggle/Colab already ship
`torch, torchvision, scipy, matplotlib, numpy`, so no install is normally needed there.

**Kaggle setup:** enable **Internet = ON** (Notebook settings) so MNIST/Frey and — if the repo
isn't attached — the exact upstream model files can download. Optionally add this repo (or just
the `models/baseline/` folder) as a **Dataset**; the next cell auto-detects it under
`/kaggle/input/`. GPU is optional (`T4 x2` works out of the box).


In [ ]:
# Local installs (Kaggle/Colab already have these). Uncomment if needed:
# %pip install torch torchvision scipy matplotlib numpy


## 1. Config — the selector

`resolve_config` gates invalid combinations. Because the pytorch backbone is the *exact*
upstream model, choosing it forces `hidden=400, latent=20, MNIST, Bernoulli`.


In [ ]:
# ============================ CONFIG ============================
BACKBONE   = "pytorch"       # "pytorch" | "novicestone"
DATASET    = "mnist"         # "mnist"   | "frey"
LIKELIHOOD = "bernoulli"     # "bernoulli" | "gaussian"
LATENT_DIM = 20
HIDDEN_DIM = 500             # NoviceStone only; paper: 500 (MNIST) / 200 (Frey)
EPOCHS     = 20
SEED       = 0
LR         = 1e-3
BATCH_SIZE = 128
# ================================================================


def resolve_config(backbone, dataset, likelihood, latent_dim, hidden_dim):
    if dataset == "frey":
        if likelihood != "gaussian":
            print("[config] Frey Face needs a Gaussian decoder -> LIKELIHOOD='gaussian'")
            likelihood = "gaussian"
        if backbone != "novicestone":
            print("[config] Frey Face needs the NoviceStone backbone -> BACKBONE='novicestone'")
            backbone = "novicestone"
    if likelihood == "gaussian" and backbone == "pytorch":
        print("[config] pytorch backbone is Bernoulli-only -> BACKBONE='novicestone'")
        backbone = "novicestone"
    if backbone == "pytorch" and (latent_dim != 20 or hidden_dim != 400):
        print("[config] exact upstream pytorch VAE is fixed at hidden=400, latent=20 -> overriding")
        latent_dim, hidden_dim = 20, 400
    return backbone, dataset, likelihood, latent_dim, hidden_dim


BACKBONE, DATASET, LIKELIHOOD, LATENT_DIM, HIDDEN_DIM = resolve_config(
    BACKBONE, DATASET, LIKELIHOOD, LATENT_DIM, HIDDEN_DIM)
print(f"backbone={BACKBONE}  dataset={DATASET}  likelihood={LIKELIHOOD}  "
      f"latent={LATENT_DIM}  hidden={HIDDEN_DIM}")


## 2. Setup — imports, seed, device


In [ ]:
import os
import sys
import ast
import glob
import math
import hashlib
import urllib.request

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset

import torchvision
from torchvision import transforms

from scipy.stats import norm
from scipy.io import loadmat

import matplotlib.pyplot as plt


def seed_everything(seed=0):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 3. Locate the baseline files + load the EXACT upstream VAE classes

The next cell finds `models/baseline/` across environments, in order:
1. a **local repo** checkout (walks up from the working directory);
2. an **uploaded Kaggle/Colab dataset** (searches `/kaggle/input/**` and `/content/**`);
3. otherwise **downloads** the exact upstream files from GitHub (needs Internet enabled).

In every case each file's **sha1 is checked against the pinned upstream hash**, so the code is
guaranteed byte-for-byte (or you get a clear drift warning). Outputs (`data/`, `results/`) are
placed in a **writable** location — the project root locally, `/kaggle/working` on Kaggle
(since `/kaggle/input` is read-only).

AST extraction then pulls only the `VAE` classes (and the pytorch `loss_function`) out of the
verbatim files, seeding the globals they need (`torch`, `nn`, `F`). Nothing else in
`pytorch_vae.py` runs — no `argparse`, no data download, no training loop.


In [ ]:
# ---- Environment detection ----
IN_KAGGLE = os.path.exists("/kaggle") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules

# Exact upstream sources + expected sha1 (byte-for-byte guarantee, incl. download fallback).
UPSTREAM = {
    "novicestone_vae.py": ("https://raw.githubusercontent.com/NoviceStone/VAE/master/models.py",
                           "06b17565437ba8bf8da07f925889e3ceaec7e0cf"),
    "pytorch_vae.py": ("https://raw.githubusercontent.com/pytorch/examples/main/vae/main.py",
                       "20986865d01324ec83085889df1955d71cbb3873"),
}


def _sha1(path):
    with open(path, "rb") as f:
        return hashlib.sha1(f.read()).hexdigest()


def _has_baseline(d):
    return bool(d) and all(os.path.exists(os.path.join(d, f)) for f in UPSTREAM)


def _writable_base():
    for p in ("/kaggle/working", "/content"):
        if os.path.isdir(p) and os.access(p, os.W_OK):
            return p
    return os.path.abspath(os.getcwd())


def _search_baseline():
    # 1) local repo checkout: walk up from cwd
    node = os.path.abspath(os.getcwd())
    cands = []
    while True:
        cands.append(os.path.join(node, "models", "baseline"))
        cands.append(os.path.join(node, "survey", "src", "related_work", "vae",
                                  "models", "baseline"))
        parent = os.path.dirname(node)
        if parent == node:
            break
        node = parent
    # 2) uploaded Kaggle/Colab dataset: find the marker file at any depth
    for pat in ("/kaggle/input/**/novicestone_vae.py", "/content/**/novicestone_vae.py"):
        cands += [os.path.dirname(m) for m in glob.glob(pat, recursive=True)]
    for d in cands:
        if _has_baseline(d):
            return d
    return None


BASELINE_DIR = _search_baseline()
if BASELINE_DIR is None:
    # 3) fallback (fresh Kaggle/Colab): download the exact upstream files (needs Internet)
    BASELINE_DIR = os.path.join(_writable_base(), "models", "baseline")
    os.makedirs(BASELINE_DIR, exist_ok=True)
    for _fname, (_url, _sha) in UPSTREAM.items():
        _dst = os.path.join(BASELINE_DIR, _fname)
        if not os.path.exists(_dst):
            print(f"downloading {_fname} <- {_url}")
            urllib.request.urlretrieve(_url, _dst)

# Integrity check + provenance.
for _fname, (_url, _sha) in UPSTREAM.items():
    _got = _sha1(os.path.join(BASELINE_DIR, _fname))
    _ok = "byte-for-byte upstream" if _got == _sha else "WARNING: differs from pinned upstream"
    print(f"  {_fname}: {_got}  ({_ok})")

# Outputs must go somewhere writable (/kaggle/input is read-only).
_vae_root = os.path.dirname(os.path.dirname(BASELINE_DIR))
if os.access(_vae_root, os.W_OK) and "/kaggle/input" not in _vae_root:
    OUT_ROOT = _vae_root
else:
    OUT_ROOT = _writable_base()
CUSTOM_DIR = os.path.join(_vae_root, "models", "custom")   # for your own models later
DATA_DIR = os.path.join(OUT_ROOT, "data")
RESULTS_DIR = os.path.join(OUT_ROOT, "results")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("environment:", "Kaggle" if IN_KAGGLE else "Colab" if IN_COLAB else "local")
print("baseline   :", BASELINE_DIR)
print("data       :", DATA_DIR)
print("results    :", RESULTS_DIR)


def _resolve(name):
    path = os.path.join(BASELINE_DIR, name)
    if not os.path.exists(path):
        raise FileNotFoundError(f"{name} not found in {BASELINE_DIR}")
    return path


def load_upstream_defs(filename, names, seed_globals):
    src = open(_resolve(filename)).read()
    tree = ast.parse(src)
    keep = [n for n in tree.body
            if isinstance(n, (ast.ClassDef, ast.FunctionDef)) and n.name in names]
    got = [n.name for n in keep]
    missing = [n for n in names if n not in got]
    if missing:
        raise RuntimeError(f"{filename}: could not find {missing} (found {got})")
    module = ast.Module(body=keep, type_ignores=[])
    ast.fix_missing_locations(module)
    ns = dict(seed_globals)
    exec(compile(module, filename, "exec"), ns)
    return {k: ns[k] for k in names}


# Exact NoviceStone VAE (byte-for-byte from baseline/novicestone_vae.py)
NoviceStoneVAE = load_upstream_defs(
    "novicestone_vae.py", ["VAE"], {"torch": torch, "nn": nn})["VAE"]

# Exact pytorch VAE + loss_function (baseline/pytorch_vae.py), scaffolding skipped
_pt = load_upstream_defs(
    "pytorch_vae.py", ["VAE", "loss_function"], {"torch": torch, "nn": nn, "F": F})
PytorchVAE = _pt["VAE"]
pt_loss_function = _pt["loss_function"]

print("NoviceStone VAE:", NoviceStoneVAE)
print("pytorch VAE:    ", PytorchVAE)
print("pytorch loss:   ", pt_loss_function)


## 4. Adapters + model factory

The two upstream classes have different `forward` signatures:

- NoviceStone: `forward(x) -> (z_mean, z_logvar, dec)`, where `dec` is a prob tensor
  (`binary`) or a `(mean, logvar)` tuple (`real`). Expects flat input.
- pytorch: `forward(x) -> (recon, mu, logvar)`; it flattens internally via `x.view(-1, 784)`.

Thin adapters normalize both so every figure routine is model-agnostic. Nothing here
reimplements the models — they only call the imported classes.


In [ ]:
def is_novicestone(model):
    return hasattr(model, "data_type")


def model_likelihood(model):
    if is_novicestone(model):
        return "gaussian" if model.data_type == "real" else "bernoulli"
    return "bernoulli"


def vae_forward(model, x):
    # Return (mu, logvar, dec) for either upstream VAE.
    if is_novicestone(model):
        mu, logvar, dec = model(x)
        return mu, logvar, dec
    recon, mu, logvar = model(x)
    return mu, logvar, recon


def dec_to_image(dec):
    # dec is a prob tensor (Bernoulli) or a (mean, logvar) tuple (Gaussian).
    return dec[0] if isinstance(dec, tuple) else dec


def build_model(backbone, input_dim, hidden_dim, latent_dim, likelihood):
    if backbone == "pytorch":
        assert (input_dim, latent_dim, likelihood) == (784, 20, "bernoulli"), \
            "exact upstream pytorch VAE is fixed to 784->400->20, MNIST/Bernoulli"
        return PytorchVAE()
    data_type = "real" if likelihood == "gaussian" else "binary"
    return NoviceStoneVAE(input_dim, hidden_dim, latent_dim, data_type)


## 5. Data — MNIST and Frey Face

Both datasets are returned as flat vectors in `[0, 1]`; `batch[0]` is always the image tensor
(MNIST's label is ignored), so training code stays dataset-agnostic.


In [ ]:
FREY_URL = "https://cs.nyu.edu/~roweis/data/frey_rawface.mat"


def load_frey(path=None):
    if path is None:
        path = os.path.join(DATA_DIR, "frey_rawface.mat")
    if not os.path.exists(path):
        print(f"Downloading Frey Face from {FREY_URL} ...")
        try:
            urllib.request.urlretrieve(FREY_URL, path)
        except Exception as e:
            raise RuntimeError(
                f"Could not download Frey Face ({e}). Download frey_rawface.mat manually "
                f"and place it at {path}.")
    ff = loadmat(path)["ff"].T.astype("float32") / 255.0     # (1965, 560), MATLAB column-major
    imgs = ff.reshape(-1, 20, 28).transpose(0, 2, 1)         # (1965, 28, 20) upright
    # If faces look transposed on your .mat, use: imgs = ff.reshape(-1, 28, 20)
    return imgs.reshape(-1, 28 * 20)


def get_dataloaders(dataset, batch_size=128, train_subset=None):
    if dataset == "mnist":
        tfm = transforms.Compose([transforms.ToTensor(),
                                  transforms.Lambda(lambda t: t.view(-1))])
        train = torchvision.datasets.MNIST(DATA_DIR, train=True, download=True, transform=tfm)
        test = torchvision.datasets.MNIST(DATA_DIR, train=False, download=True, transform=tfm)
        input_dim, img_shape = 784, (28, 28)
    elif dataset == "frey":
        X = torch.from_numpy(load_frey())
        n_test = 200
        train = TensorDataset(X[:-n_test])
        test = TensorDataset(X[-n_test:])
        input_dim, img_shape = 560, (28, 20)
    else:
        raise ValueError(f"unknown dataset {dataset!r}")

    if train_subset is not None:
        train = Subset(train, list(range(min(train_subset, len(train)))))
    train_loader = DataLoader(train, batch_size=batch_size, shuffle=True, drop_last=True)
    test_loader = DataLoader(test, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader, input_dim, img_shape


## 6. ELBO, training, IW likelihood, tiling

Per-datapoint `ELBO = log p(x|z) - KL`. The reported metric (test ELBO/datapoint) is computed
the same way for both models so Figure 2 is comparable; the pytorch path *trains* with its own
upstream `loss_function` (summed BCE+KL), the NoviceStone path with the mean negative ELBO.


In [ ]:
def elbo_per_point(x, mu, logvar, dec, likelihood):
    if likelihood == "bernoulli":
        log_pxz = -F.binary_cross_entropy(dec, x, reduction="none").sum(1)
    else:
        mean, lv = dec
        log_pxz = -0.5 * (math.log(2 * math.pi) + lv + (x - mean) ** 2 / lv.exp()).sum(1)
    kl = 0.5 * (mu.pow(2) + logvar.exp() - 1.0 - logvar).sum(1)
    return log_pxz - kl


def training_loss(model, x):
    if is_novicestone(model):
        mu, logvar, dec = model(x)
        return -elbo_per_point(x, mu, logvar, dec, model_likelihood(model)).mean()
    recon, mu, logvar = model(x)         # exact upstream forward
    return pt_loss_function(recon, x, mu, logvar)   # exact upstream loss (summed)


@torch.no_grad()
def eval_elbo(model, loader, device, n_batches=None):
    model.eval()
    lk = model_likelihood(model)
    total, count = 0.0, 0
    for i, batch in enumerate(loader):
        if n_batches is not None and i >= n_batches:
            break
        x = batch[0].to(device)
        mu, logvar, dec = vae_forward(model, x)
        total += elbo_per_point(x, mu, logvar, dec, lk).sum().item()
        count += x.size(0)
    return total / max(count, 1)


def train(model, train_loader, test_loader, epochs, lr, device,
          eval_every=5000, eval_batches=20, verbose=True):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"seen": [], "test_elbo": []}
    seen, next_eval = 0, 0
    for ep in range(epochs):
        model.train()
        for batch in train_loader:
            x = batch[0].to(device)
            opt.zero_grad()
            loss = training_loss(model, x)
            loss.backward()
            opt.step()
            seen += x.size(0)
            if seen >= next_eval:
                history["seen"].append(seen)
                history["test_elbo"].append(eval_elbo(model, test_loader, device, eval_batches))
                next_eval += eval_every
                model.train()
        if verbose:
            te = history["test_elbo"][-1] if history["test_elbo"] else float("nan")
            print(f"  epoch {ep + 1}/{epochs}  test ELBO/point = {te:.2f}")
    return model, history


@torch.no_grad()
def iw_loglik(model, loader, device, L=64, n_datapoints=1000):
    model.eval()
    lk = model_likelihood(model)
    total, count = 0.0, 0
    for batch in loader:
        x = batch[0].to(device)
        mu, logvar = model.encode(x)
        var = logvar.exp()
        logws = []
        for _ in range(L):
            z = model.reparameterize(mu, logvar)
            dec = model.decode(z)
            if lk == "bernoulli":
                log_pxz = -F.binary_cross_entropy(dec, x, reduction="none").sum(1)
            else:
                mean, lv = dec
                log_pxz = -0.5 * (math.log(2 * math.pi) + lv + (x - mean) ** 2 / lv.exp()).sum(1)
            log_pz = -0.5 * (z ** 2 + math.log(2 * math.pi)).sum(1)
            log_qz = -0.5 * ((z - mu) ** 2 / var + logvar + math.log(2 * math.pi)).sum(1)
            logws.append(log_pxz + log_pz - log_qz)
        logw = torch.stack(logws, dim=1)
        ll = torch.logsumexp(logw, dim=1) - math.log(L)
        total += ll.sum().item()
        count += x.size(0)
        if count >= n_datapoints:
            break
    return total / max(count, 1)


def tile_images(images, grid_shape, img_shape, spacing=1, bg=0.0):
    images = np.asarray(images, dtype=np.float32)
    H, W = img_shape
    if images.ndim == 2:
        images = images.reshape(-1, H, W)
    rows, cols = grid_shape
    canvas = np.full((rows * H + (rows - 1) * spacing,
                      cols * W + (cols - 1) * spacing), bg, dtype=np.float32)
    for idx in range(min(len(images), rows * cols)):
        r, c = divmod(idx, cols)
        y, x = r * (H + spacing), c * (W + spacing)
        canvas[y:y + H, x:x + W] = images[idx]
    return canvas


def show_reconstructions(model, loader, img_shape, n=8):
    model.eval()
    x = next(iter(loader))[0][:n].to(device)
    with torch.no_grad():
        _, _, dec = vae_forward(model, x)
        recon = dec_to_image(dec)
    top = tile_images(x.cpu().numpy(), (1, n), img_shape)
    bot = tile_images(recon.cpu().numpy(), (1, n), img_shape)
    canvas = np.concatenate([top, np.zeros((2, top.shape[1])), bot], axis=0)
    plt.figure(figsize=(n, 2.6))
    plt.imshow(canvas, cmap="gray")
    plt.axis("off")
    plt.title("top: input     bottom: reconstruction")
    plt.show()


## 7. Quick single-model demo (uses the CONFIG above)

Trains one model from the selector (few epochs) and shows reconstructions.


In [ ]:
_tr, _te, _in, _img = get_dataloaders(DATASET, BATCH_SIZE)
seed_everything(SEED)
demo_model = build_model(BACKBONE, _in, HIDDEN_DIM, LATENT_DIM, LIKELIHOOD)
demo_model, _ = train(demo_model, _tr, _te, epochs=min(EPOCHS, 3), lr=LR, device=device)
show_reconstructions(demo_model, _te, _img)


## Figure 2 — lower bound vs. amount of training

NoviceStone across several `Nz`, plus one curve from the exact pytorch VAE (fixed at Nz=20).
Same axes as the paper; the pytorch-vs-NoviceStone overlay replaces "AEVB vs wake-sleep".


In [ ]:
def figure2(latent_dims=(3, 10, 20), epochs=5, hidden_dim=500, eval_every=5000,
            include_pytorch=True):
    tr, te, input_dim, _ = get_dataloaders("mnist", BATCH_SIZE)
    plt.figure(figsize=(8, 6))
    for nz in latent_dims:
        seed_everything(SEED)
        m = build_model("novicestone", input_dim, hidden_dim, nz, "bernoulli")
        print(f"[fig2] novicestone Nz={nz}")
        m, h = train(m, tr, te, epochs, LR, device, eval_every=eval_every, verbose=False)
        plt.plot(h["seen"], h["test_elbo"], "--", label=f"NoviceStone, Nz={nz}")
    if include_pytorch:
        seed_everything(SEED)
        m = build_model("pytorch", 784, 400, 20, "bernoulli")
        print("[fig2] pytorch Nz=20")
        m, h = train(m, tr, te, epochs, LR, device, eval_every=eval_every, verbose=False)
        plt.plot(h["seen"], h["test_elbo"], "-", linewidth=2, label="pytorch, Nz=20 (fixed)")
    plt.xlabel("# training points evaluated")
    plt.ylabel("test ELBO (nats / datapoint)")
    plt.title("Figure 2 — variational lower bound (MNIST)")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "figure2.png"), dpi=120)
    plt.show()


figure2(latent_dims=(3, 10, 20), epochs=5)


## Figure 3 — marginal log-likelihood, small vs. full training set

Importance-weighted marginal log-likelihood on held-out data, `Nz=3` (NoviceStone, since the
pytorch model is fixed at 20). The small subset overfits relative to the full set.


In [ ]:
def figure3(train_sizes=(1000, None), epochs=8, hidden_dim=500, latent_dim=3, L=32):
    plt.figure(figsize=(8, 6))
    for ts in train_sizes:
        tr, te, input_dim, _ = get_dataloaders("mnist", BATCH_SIZE, train_subset=ts)
        seed_everything(SEED)
        m = build_model("novicestone", input_dim, hidden_dim, latent_dim, "bernoulli").to(device)
        opt = torch.optim.Adam(m.parameters(), lr=LR)
        seen, seens, lls = 0, [], []
        for ep in range(epochs):
            m.train()
            for batch in tr:
                x = batch[0].to(device)
                opt.zero_grad()
                loss = training_loss(m, x)
                loss.backward()
                opt.step()
                seen += x.size(0)
            ll = iw_loglik(m, te, device, L=L, n_datapoints=1000)
            seens.append(seen)
            lls.append(ll)
            print(f"[fig3] train={ts or 'full'} epoch {ep + 1}/{epochs} logp(x)~{ll:.2f}")
        plt.plot(seens, lls, marker="o", label=f"train set = {ts or 'full'}")
    plt.xlabel("# training points evaluated")
    plt.ylabel("marginal log-likelihood (IW estimate, nats)")
    plt.title(f"Figure 3 — marginal likelihood, Nz={latent_dim}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "figure3.png"), dpi=120)
    plt.show()


figure3(train_sizes=(1000, None), epochs=8)


## Figure 4 — learned 2-D manifold

NoviceStone with `Nz=2`, decoded on an inverse-Gaussian-CDF grid (`scipy.stats.norm.ppf`).
`figure4("frey")` uses the Gaussian decoder; `figure4("mnist")` the Bernoulli decoder.


In [ ]:
def figure4(dataset, hidden_dim=500, epochs=20, grid=20):
    likelihood = "gaussian" if dataset == "frey" else "bernoulli"
    tr, te, input_dim, img_shape = get_dataloaders(dataset, BATCH_SIZE)
    seed_everything(SEED)
    m = build_model("novicestone", input_dim, hidden_dim, 2, likelihood)
    print(f"[fig4] {dataset} / {likelihood}, Nz=2")
    m, _ = train(m, tr, te, epochs, LR, device, verbose=False)
    m.eval()
    zg = norm.ppf(np.linspace(0.05, 0.95, grid)).astype("float32")
    zs = np.array([[zx, zy] for zy in zg for zx in zg], dtype="float32")
    with torch.no_grad():
        dec = m.decode(torch.from_numpy(zs).to(device))
        imgs = dec_to_image(dec).cpu().numpy()
    canvas = tile_images(imgs, (grid, grid), img_shape, spacing=1)
    plt.figure(figsize=(7, 7 * img_shape[0] / img_shape[1]))
    plt.imshow(canvas, cmap="gray")
    plt.axis("off")
    plt.title(f"Figure 4 — learned 2-D manifold ({dataset})")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"figure4_{dataset}.png"), dpi=120)
    plt.show()


figure4("mnist", epochs=20, grid=20)
# figure4("frey", epochs=40, grid=16)   # panel (a); needs frey_rawface.mat


## Figure 5 — random samples for different latent sizes

NoviceStone trained at each `Nz`, then `z ~ N(0, I)` decoded and tiled. (The exact pytorch VAE
could only fill the Nz=20 panel.)


In [ ]:
def figure5(latent_dims=(2, 5, 10, 20), epochs=10, hidden_dim=500, n_samples=100,
            dataset="mnist"):
    likelihood = "gaussian" if dataset == "frey" else "bernoulli"
    tr, te, input_dim, img_shape = get_dataloaders(dataset, BATCH_SIZE)
    side = int(round(math.sqrt(n_samples)))
    fig, axes = plt.subplots(1, len(latent_dims), figsize=(4 * len(latent_dims), 4))
    axes = np.atleast_1d(axes)
    for ax, nz in zip(axes, latent_dims):
        seed_everything(SEED)
        m = build_model("novicestone", input_dim, hidden_dim, nz, likelihood)
        print(f"[fig5] {dataset} Nz={nz}")
        m, _ = train(m, tr, te, epochs, LR, device, verbose=False)
        m.eval()
        with torch.no_grad():
            z = torch.randn(n_samples, nz, device=device)
            imgs = dec_to_image(m.decode(z)).cpu().numpy()
        ax.imshow(tile_images(imgs, (side, side), img_shape, spacing=1), cmap="gray")
        ax.axis("off")
        ax.set_title(f"Nz={nz}")
    fig.suptitle(f"Figure 5 — random samples ({dataset})")
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTS_DIR, "figure5.png"), dpi=120)
    plt.show()


figure5(latent_dims=(2, 5, 10, 20), epochs=10)


## Notes

- **Switch implementations**: set `BACKBONE` in the config cell. `pytorch` is the exact
  upstream model (fixed 784→400→20, MNIST/Bernoulli); `novicestone` is parameterizable and
  also does Frey Face (Gaussian).
- **Frey manifold (Fig 4a)**: run `figure4("frey")` — needs `data/frey_rawface.mat`.
- **Closer to the paper**: raise `EPOCHS`, set `hidden_dim=500` (MNIST) / `200` (Frey), add
  `Nz=200` to `figure2`, and raise `L` in the IW estimator.
- The two upstream files in `models/baseline/` are byte-for-byte from their repos — the loader
  cell prints each file's sha1 and flags any drift from the pinned hashes.
- Outputs go under the auto-detected `OUT_ROOT`: the project root locally, `/kaggle/working`
  on Kaggle. Datasets land in `data/`, figures in `results/`. Your own models can go in
  `models/custom/` (path exposed as `CUSTOM_DIR`).
- **Kaggle**: Internet = ON (for MNIST/Frey + the download fallback). Either attach the repo as
  a Dataset (auto-detected under `/kaggle/input/`) or let the loader fetch the exact upstream
  files. Everything written to `/kaggle/working` is saved as the notebook's output.
